# Introduction to Sets in Java and the Class `RecursiveSet`

In theoretical computer science, the concept of a *Set* is fundamental to formal models such as automata, grammars, constraint satisfaction problems (CSPs like Sudoku), and propositional logic. Before we can implement algorithms for these structures, we must establish a computationally sound representation of mathematical sets.

## Mathematical Definition
A set is a collection of distinct objects where the order of elements is irrelevant. Formally, for two sets $A$ and $B$, the following holds true:

$$ A = \{1, 2, 3\} $$
$$ B = \{3, 2, 1\} $$
$$ A = B $$

Furthermore, adding an element that already exists in the set does not alter the set. If we define $C = A \cup \{2\}$, then $C$ evaluates to $\{1, 2, 3\}$.

In the following we will examine how these mathematical properties translate to data structures in Java.

This notebook has two parts.  The first part discusses the sets provided by Java, while the second part introduces the class `RecursiveSet`, which is used throughout these lecture notes.  This class is defined in the file `../lib/RecursiveSet.jsh`.  As this file contains `import` statements, it has to be loaded *before* any classes are defined.  Therefore, we load it now.  The magic command `%load` reads a file and evaluates its contents.

In [ ]:
%load ../lib/RecursiveSet.jsh

Java provides the interface `Set`.  The easiest way to create a set is to use the static method `Set.of`.

In [ ]:
var A = Set.of(1, 2, 3);
A

In [ ]:
var B = Set.of(3, 2, 1);
B

Note that Java prints sets using *square brackets*.  Furthermore, the order in which the elements are printed is unspecified.

The operator `==` checks whether `A` and `B` are the *same object*, i.e. it compares the *references* to the objects `A` and `B`.  As `A` and `B` have been created independently, they are different objects.

In [ ]:
A == B

In order to check whether two objects have the same *value*, Java provides the method `equals`.  For sets, `equals` returns `true` if both sets contain the same elements.

In [ ]:
A.equals(B)

Arrays are different: Arrays do not redefine the method `equals`.  Therefore, `equals` compares the references of two arrays.

In [ ]:
int[] LA = {1, 2, 3};
int[] LB = {1, 2, 3};
LA.equals(LB)

In order to compare the contents of two arrays, we have to use the static method `Arrays.equals`.

In [ ]:
Arrays.equals(LA, LB)

Lists, on the other hand, are compared by value.

In [ ]:
List.of(1, 2, 3).equals(List.of(1, 2, 3))

## Native Java Sets

The interface `Set` has several implementations.  The most important ones are `HashSet`, `LinkedHashSet`, and `TreeSet`.  A `HashSet` uses the methods `hashCode` and `equals` of its elements to enforce the mathematical property of uniqueness.

In [ ]:
Set<Integer> primitiveSet = new HashSet<>();

primitiveSet.add(1);
primitiveSet.add(2);
primitiveSet.add(3);
primitiveSet.add(2);

System.out.println("Primitive set contents: " + primitiveSet);
System.out.println("Size of the set: " + primitiveSet.size());

### Sets of Sets

Theoretical computer science frequently requires more complex structures like *sets that contain other sets* (e.g., powerset constructions or partitions).

Let us observe what happens when we nest `HashSet` objects. In the following example, we create three sets. We add the number `1` to two distinct sets, and then add both of these sets to a main set.

In [ ]:
Set<Set<Integer>> mainSet = new HashSet<>();
mainSet

In [ ]:
Set<Integer> childSetA = new HashSet<>();
childSetA.add(1);
childSetA

In [ ]:
Set<Integer> childSetB = new HashSet<>();
childSetB.add(1);
childSetB

In [ ]:
// Adding structurally identical sets to the main set
mainSet.add(childSetA);
mainSet.add(childSetB);

System.out.println("Main set contents: " + mainSet);
System.out.println("Size of the main set: " + mainSet.size());

### Analysis of the Output
The `mainSet` contains only one child set and its size is `1`.  This is mathematically correct, since $\{ \{1\}, \{1\} \}$ is the same set as $\{ \{1\} \}$.

The reason is that `HashSet` does not compare its elements by their *references*.  Instead, it uses the methods `hashCode` and `equals` of the elements.  As the interface `Set` specifies that two sets are equal if they contain the same elements, `childSetA` and `childSetB` are considered to be equal, although they are different objects:

In [ ]:
childSetA == childSetB

In [ ]:
childSetA.equals(childSetB)

This behaviour is called **Value Semantics** (also known as Deep Equality): Two structures are considered equal if their mathematical contents are identical, regardless of their memory location.  In this respect, Java is better than JavaScript and TypeScript, where the native sets compare objects by reference.

### The Pitfall: Mutable Elements

However, there is still a problem.  When an object is inserted into a `HashSet`, its hash code is computed and the object is stored at a location that depends on this hash code.  What happens if we change `childSetA` after it has been inserted into `mainSet`?

In [ ]:
childSetA.add(2);
System.out.println("Main set contents: " + mainSet);

The main set seems to contain the set $\{1, 2\}$ now.  Let us check this.

In [ ]:
mainSet.contains(childSetA)

In [ ]:
mainSet.contains(Set.of(1, 2))

The `mainSet` has been silently corrupted:  The set $\{1,2\}$ is printed as an element of `mainSet`, but `mainSet` does not find it, because it has been stored at the location corresponding to the hash code of the set $\{1\}$.

In mathematics, a set like $\{1\}$ is a constant.  It cannot suddenly become $\{1,2\}$.  Therefore, **an object must never be changed after it has been inserted into a set**.  Java offers *immutable* sets for this purpose.  The sets created by `Set.of` are immutable: Any attempt to change them throws an exception.

In [ ]:
var frozenSet = Set.of(1, 2);
frozenSet.add(3);

Unfortunately, the method `Set.of` has another surprising property: It refuses to create a set from a list of arguments that contains duplicates.

In [ ]:
Set.of(1, 1)

## Representing Mathematical Tuples

In addition to sets, theoretical models frequently rely on **Tuples**. A tuple is a finite, ordered sequence of elements. Unlike sets, the order of elements in a tuple is strictly preserved, and duplicate elements are allowed.

The most common example of a tuple, which you already know from high school mathematics, is a **Coordinate Pair** $(x, y)$ in a 2D Cartesian coordinate system. 

For two points $P$ and $Q$:
$$ P = (2, 5) $$
$$ Q = (5, 2) $$

Because the order matters, $P \neq Q$. The $x$-coordinate and $y$-coordinate have strictly defined positions. 

Let us explore how to represent such coordinate tuples in Java, and why we might want to store them in a Set (e.g., a set of visited coordinates on a grid).

## Using Arrays to Represent Tuples

A naive approach is to use Java arrays to represent these coordinate pairs, since arrays preserve the order of their elements.

Let us define a set of coordinates that a robot has already visited, and then attempt to check if a specific coordinate is in that set.

In [ ]:
Set<int[]> visitedCoordinates = new HashSet<>();

int[] startPoint = {2, 5};
visitedCoordinates.add(startPoint);
visitedCoordinates

The output is not very helpful: Java does not print the contents of an array.  Instead, it prints the type `[I` (array of `int`) together with the hash code of the array.

In [ ]:
// The robot wants to check if it has already visited the point (2, 5)
int[] currentPosition = {2, 5};
boolean hasVisited    = visitedCoordinates.contains(currentPosition);

System.out.println("Has the robot visited (2, 5) before? -> " + hasVisited);

### Analysis of the Output
The native data structures fail us. The call `visitedCoordinates.contains(currentPosition)` evaluates to `false`, even though the point `{2, 5}` was clearly added to the set.

As discussed above, arrays do not redefine the methods `equals` and `hashCode`.  Hence, they are compared by **Reference Equality**. The computer treats `startPoint` and `currentPosition` as two completely distinct objects, even though they represent the exact same mathematical point.

In [ ]:
startPoint == currentPosition

In [ ]:
startPoint.equals(currentPosition)

Furthermore, arrays introduce a second, highly dangerous flaw for mathematical modeling: **Mutability**.
In mathematics, the point $(2, 5)$ is a constant, abstract concept. The number $2$ cannot suddenly become $99$. However, in Java, arrays can be altered after their creation.

In [ ]:
// Arrays are mutable (changeable)
startPoint[0] = 99;
Arrays.toString(startPoint)

In [ ]:
for (int[] point : visitedCoordinates) {
    System.out.println(Arrays.toString(point));
}

If we use arrays to represent coordinates, an accidental modification elsewhere in the code could silently corrupt our entire logic. We need a structure that provides **Deep Equality** (so it works inside Sets/Maps) and **Immutability** (so it cannot be altered).

## Records

Java provides *records*.  A record is a class that is used to store immutable data.  For example, the declaration
```java
record Point(int x, int y) {}
```
declares a class `Point` with two components `x` and `y`.  The compiler automatically generates
1. a constructor `Point(int x, int y)`,
2. the methods `x()` and `y()` that return the values of the components,
3. the methods `equals` and `hashCode` that compare records structurally (Deep Equality), and
4. a method `toString`.

As the components of a record are `final`, a record can not be changed once it has been created.

In [ ]:
record Point(int x, int y) {}

In [ ]:
// Create a mathematically sound set of visited coordinates
Set<Point> safeVisitedCoords = new HashSet<>();

var p1 = new Point(2, 5);
safeVisitedCoords.add(p1);
safeVisitedCoords

In [ ]:
// We search using a completely new instance with the same mathematical content
var searchPoint     = new Point(2, 5);
var isPointFound    = safeVisitedCoords.contains(searchPoint);

System.out.println("Robot has visited: " + safeVisitedCoords);
System.out.println("Has the robot visited (2, 5) before? -> " + isPointFound);

The components of a record are stored in fields that are both `private` and `final`.  Therefore, trying to change a component of a record results in a compile time error.

In [ ]:
p1.x = 3;

The components can only be read using the generated accessor methods.

In [ ]:
p1.x()

In [ ]:
safeVisitedCoords

## Associating Data: Maps

Now that we know how to define strictly correct coordinates using records, we often want to associate data with these coordinates. In computer science, a data structure that associates a *Key* with a *Value* is called a **Map** (or Dictionary / Associative Array).

Think of a chess board or a grid: You have a coordinate (the Key) and a piece on that coordinate (the Value). Mathematically, this is a mapping:
$$ f: (x, y) \rightarrow \text{Value} $$
$$ f((2, 5)) = \text{"Queen"} $$

Let us first try to implement this mapping using arrays as keys.

In [ ]:
Map<int[], String> nativeGrid = new HashMap<>();

int[] posP = {2, 5};
nativeGrid.put(posP, "Queen");

// We try to look up what piece is at coordinate (2, 5)
int[] searchPos = {2, 5};
String pieceAtPos = nativeGrid.get(searchPos);

System.out.println("Piece at (2, 5) -> " + pieceAtPos);

### Analysis of the Output
The output is `null`. Once again, the lookup fails because arrays are compared by **Reference Equality**.  The map searches for the exact object `searchPos`, not for the mathematical sequence `(2, 5)`.

As a `HashMap` uses the methods `hashCode` and `equals` of its keys, everything works as expected if we use records as keys.

In [ ]:
// We define a map where the key is a Point and the value is a String
Map<Point, String> safeGrid = new HashMap<>();

safeGrid.put(new Point(2, 5), "Queen");

// Searching with a new, structurally identical point
var lookupPoint = new Point(2, 5);
var foundPiece  = safeGrid.get(lookupPoint);

System.out.println("Safe grid contents: " + safeGrid);
System.out.println("Piece at (2, 5) -> " + foundPiece);

**Summary:**
1. **Equality:** Use the method `equals` to compare objects by value.  The operator `==` compares references.
2. **Sets:** The sets provided by Java compare their elements by value.  However, an element must never be modified after it has been inserted into a set.
3. **Tuples:** Use records when you need an ordered, immutable pair or sequence (e.g., coordinates, transition pairs).  Do not use arrays as elements of sets or as keys of maps.
4. **Mappings:** A `HashMap` works correctly if its keys implement `equals` and `hashCode` structurally, e.g. if the keys are records or immutable sets.

# The Class `RecursiveSet`

Although the sets provided by Java have value semantics, they are not very convenient for our purposes:
1. Mutable sets that are elements of other sets can silently corrupt these sets.
2. The set-theoretic operations are clumsy.  For example, to compute the union of two sets $X$ and $Y$ we have to write
   ```java
   Set<Integer> U = new HashSet<>(X);
   U.addAll(Y);
   ```
   as there is no method that returns the union of two sets.
3. Sets are printed using square brackets instead of curly braces and the order of the elements is unpredictable.
4. There are no methods to compute the power set or the cartesian product of two sets.

Therefore, we use the class `RecursiveSet`, which is defined in the file `../lib/RecursiveSet.jsh` that we have loaded at the beginning of this notebook.  This class is the Java counterpart of the TypeScript library `recursive-set`.  It implements the interface `Set`, so all the methods of `Set` are available.  In addition, it offers the operations discussed below.  In the following notebooks, the file `RecursiveSet.jsh` will always be loaded in the first cell.

The constructor `new RecursiveSet<>(a, b, c)` creates the set $\{a, b, c\}$.  Duplicates are allowed and are silently removed.  Sets are printed using curly braces and the elements are listed in the order they have been inserted.

In [ ]:
var recursiveMainSet = new RecursiveSet<RecursiveSet<Integer>>();

var recursiveChildA = new RecursiveSet<>(1);
var recursiveChildB = new RecursiveSet<>(1, 1);

recursiveMainSet.add(recursiveChildA);
recursiveMainSet.add(recursiveChildB);

System.out.println("RecursiveSet contents: " + recursiveMainSet);
System.out.println("Size of the RecursiveSet: " + recursiveMainSet.size());

# Advanced Operations with `RecursiveSet`

Now that we have established how to construct mathematically sound sets, we can perform actual computations with them. In theoretical computer science, we rarely just store elements; we manipulate them using set theory and functional programming.

## Basic Mutations: Add and Remove

Adding and removing elements from a `RecursiveSet` works exactly as expected.

In [ ]:
var alphabet = new RecursiveSet<>("a", "b", "c");

System.out.println("Initial alphabet: " + alphabet);

// Removing an element
alphabet.remove("b");
System.out.println("After removing 'b': " + alphabet);

// Adding an element
alphabet.add("d");
System.out.println("After adding 'd': " + alphabet);

## Immutability (The Freeze Contract)

In mathematics, a set like $A = \{1, 2\}$ is a constant concept. If you use set $A$ as an element inside another set $B$, $A$ **must never change** again. If you were allowed to add a $3$ to $A$ later, it would silently corrupt the structure of $B$.

To prevent this dangerous behavior, `RecursiveSet` implements a strict **Freeze Contract**. The moment you insert a set into another collection (like a `RecursiveSet`, a `HashSet`, or a `HashMap`), it becomes **frozen** (immutable).  Technically, a `RecursiveSet` is frozen as soon as its hash code is computed.

In [ ]:
var stateA   = new RecursiveSet<>(1, 2);
var powerset = new RecursiveSet<RecursiveSet<Integer>>();

// By adding stateA to the powerset, stateA is permanently FROZEN!
powerset.add(stateA);
powerset

In [ ]:
// Attempting to mutate a frozen set will throw an exception
stateA.add(3);

What if we actually *need* to modify a set that has been frozen? We cannot change the original, because it is now securely locked inside the `powerset`. However, we can create a shallow, mutable copy of it using the `clone()` method.

In [ ]:
var stateAModified = stateA.clone();
stateAModified.add(3);

System.out.println("Original stateA (still frozen inside powerset): " + stateA);
System.out.println("New stateAModified (unfrozen and independent): " + stateAModified);

In [ ]:
powerset

The freeze contract also protects a `RecursiveSet` that is inserted into one of Java's own collections.

In [ ]:
var child = new RecursiveSet<>(1);
Set<Set<Integer>> javaSet = new HashSet<>();
javaSet.add(child);
child.isFrozen()

## Mathematical Set Operations

The library provides all standard mathematical set operations built-in. These operations always return a **new** `RecursiveSet`, leaving the original sets unmodified.

- **Union** ($A \cup B$): Elements in $A$ or $B$.
- **Intersection** ($A \cap B$): Elements in both $A$ and $B$.
- **Difference** ($A \setminus B$): Elements in $A$ but not in $B$.
- **Symmetric Difference** ($A \bigtriangleup B$): Elements in either $A$ or $B$, but not in both.

In [ ]:
var setX = new RecursiveSet<>(1, 2, 3);
var setY = new RecursiveSet<>(3, 4, 5);

var unionSet        = setX.union(setY);
var intersectionSet = setX.intersection(setY);
var differenceSet   = setX.difference(setY);          // X without Y
var symDiffSet      = setX.symmetricDifference(setY); // (X ∪ Y) \ (X ∩ Y)

System.out.println("X = " + setX + ", Y = " + setY);
System.out.println("X ∪ Y (Union): " + unionSet);
System.out.println("X ∩ Y (Intersection): " + intersectionSet);
System.out.println("X \\ Y (Difference): " + differenceSet);
System.out.println("X △ Y (Symmetric Difference): " + symDiffSet);

### Subset and Superset Relations ($\subseteq$, $\supseteq$)
Two further fundamental relations in set theory are:
- $A \subseteq B$ (*A is a subset of B*): every element of $A$ is also contained in $B$.
- $A \supseteq B$ (*A is a superset of B*): $A$ contains all elements of $B$.

Note that $A = B$ implies both $A \subseteq B$ and $A \supseteq B$ simultaneously.

In [ ]:
var small = new RecursiveSet<>(2, 3);
var large = new RecursiveSet<>(1, 2, 3, 4, 5);

System.out.println(small + " ⊆ " + large + ": " + small.isSubset(large));
System.out.println(large + " ⊇ " + small + ": " + large.isSuperset(small));
System.out.println(large + " ⊆ " + small + ": " + large.isSubset(small));

### Selecting a Random Element
In mathematical proofs, we frequently state: *"Let $x$ be a random element from the set $A$."* 
If you attempt to simulate this in code by iterating over the set and stopping after the first element, you will always get the exact same element. This is because `RecursiveSet` internally preserves the insertion order. To genuinely select a random element, you must use the `pickRandom()` method.

In [ ]:
setX

In [ ]:
System.out.println("X = " + setX);
for (int i = 0; i < 5; ++i) {
    System.out.println("Random element x ∈ X: " + setX.pickRandom());
}

## Cartesian Product and Powerset

In addition to basic operations, set theory defines operations that generate entirely new, structurally distinct sets. You may recall these concepts from high school combinatorics or probability theory.

### Cartesian Product ($A \times B$)
The Cartesian product combines two sets into a set of all possible ordered pairs (Tuples), where the first element belongs to $A$ and the second belongs to $B$. 

A classic everyday example is finding all possible combinations of two independent choices, such as pairing a main dish with a drink from a menu.

In [ ]:
var mainDishes = new RecursiveSet<>("Burger", "Pasta");
var drinks     = new RecursiveSet<>("Water", "Cola");

var mealCombinations = mainDishes.cartesianProduct(drinks);

System.out.println("Cartesian Product (A x B):");
mealCombinations

The ordered pairs are objects of the record class `Pair`, which is defined in the file `RecursiveSet.jsh` as follows:
```java
record Pair<A, B>(A first, B second) { ... }
```
The components of a pair are accessed via the methods `first()` and `second()`.

In [ ]:
for (var meal : mealCombinations) {
    System.out.println(meal.first() + " with " + meal.second());
}

### Powerset ($\mathcal{P}(A)$ or $2^A$)
The powerset is the set of all possible subsets of $A$, including the empty set ($\emptyset$) and the set $A$ itself. The size of the powerset is always $2^{|A|}$.

A common real-world analogy is choosing optional toppings for a pizza. Given a set of available toppings, you can choose none (the empty set), exactly one, any combination, or all of them. The powerset represents every possible customized pizza you could order.

In [ ]:
var optionalToppings  = new RecursiveSet<>("Cheese", "Sauce", "Olives");
var allPossibleOrders = optionalToppings.powerset();

System.out.println("Powerset P(A):");
System.out.println(allPossibleOrders);
System.out.println("Size of the Powerset: " + allPossibleOrders.size()); // Expected: 2^3 = 8

*Warning:* The powerset grows exponentially. A set with just 20 elements will generate a powerset with $1,048,576$ subsets. The class `RecursiveSet` will intentionally throw an exception if you attempt to compute a powerset for $|A| > 20$ to prevent your computer from running out of memory.

## Mathematical View of Functional Operations

Before using functional operations in Java, it is helpful to describe them mathematically.

Let $A$ be a set, let $f : A \to B$ be a function, and let $P(x)$ be a predicate.
A predicate is a statement about an element $x$ that is either true or false.

In Java, a *lambda expression* such as `word -> word.length() == 3`
defines a predicate $P(word)$.
A lambda expression such as `word -> word.toUpperCase()`
defines a function $f(word)$.

### Set-builder notation
In mathematics, we often describe sets by specifying a property that their elements must satisfy:

$$
\{x \in A \mid P(x)\}
$$

This is read as:
“the set of all elements $x$ in $A$ such that $P(x)$ holds.”

### `filter(predicate)`
The operation `filter` keeps exactly those elements that satisfy the predicate:

$$
A\mathrm{.filter}(P) = \{x \in A \mid P(x)\}
$$

In [ ]:
var vocabulary = new RecursiveSet<>("cat", "dog", "bat", "mouse", "owl");

var threeLetterWords = vocabulary.filter(word -> word.length() == 3);

System.out.println("Original vocabulary: " + vocabulary);
System.out.println("Filtered (length == 3): " + threeLetterWords);

### `map(transform)`
The operation `map` applies a function to every element of a set:

$$
A\mathrm{.map}(f) = \{f(x) \mid x \in A\}
$$

Note that the result is again a set, so duplicate results are automatically removed.

In [ ]:
"abc".toUpperCase()

In [ ]:
// We transform every word to uppercase
var uppercaseWords = threeLetterWords.map(word -> word.toUpperCase());
System.out.println("Uppercase words: " + uppercaseWords);

In [ ]:
// Example of duplicate reduction during map:
// We map every word to its length. ("cat" -> 3, "bat" -> 3)
var wordLengths = threeLetterWords.map(word -> word.length());
System.out.println("Word lengths set: " + wordLengths); // Expected: {3}

### `every(predicate)` and `some(predicate)`
These two operations correspond directly to logical quantifiers:

$$
A\mathrm{.every}(P) \iff \forall x \in A: P(x)
$$

$$
A\mathrm{.some}(P) \iff \exists x \in A: P(x)
$$

In [ ]:
var numbers = new RecursiveSet<>(2, 4, 6, 8);

var areAllEven    = numbers.every(n -> n % 2 == 0);
var containsEight = numbers.some(n -> n == 8);
var containsOdd   = numbers.some(n -> n % 2 != 0);

System.out.println("Are all numbers even? -> " + areAllEven);
System.out.println("Does the set contain an 8? -> " + containsEight);
System.out.println("Does the set contain an odd number? -> " + containsOdd);

### `filterMap(predicate, transform)`
This operation combines filtering and mapping in a single step:

$$
A\mathrm{.filterMap}(P, f) = \{f(x) \mid x \in A \land P(x)\}
$$

In [ ]:
var mixedNumbers = new RecursiveSet<>(1, 2, 3, 4, 5);

// filterMap: Keep only odd numbers AND multiply them by 10
var oddTens = mixedNumbers.filterMap(
    n -> n % 2 == 1,    // Predicate (filter)
    n -> n * 10         // Transform (map)
);
System.out.println("Odd numbers multiplied by 10: " + oddTens);

### `flatMap(mapper)`
If each element $x$ is mapped to an entire set $f(x)$, then `flatMap` collects all of these sets and merges them into one:

$$
A\mathrm{.flatMap}(f) = \bigcup_{x \in A} f(x)
$$

In [ ]:
var smallNumbers = new RecursiveSet<>(1, 2, 3);

// For every natural number n, we generate a set containing n and -n.
smallNumbers.flatMap(n -> new RecursiveSet<>(n, -n))

The file `RecursiveSet.jsh` also defines a *function* `flatMap(items, mapper)`.  Here, the first argument `items` can be any collection, e.g. a list.  The function computes the union of all sets `mapper(x)` where `x` runs through `items`.

In [ ]:
// We can use a simple list as input to generate new elements
var naturalNumbers = List.of(1, 2, 3);

// For every natural number n, we generate a set containing n and -n.
var S = flatMap(naturalNumbers, n -> new RecursiveSet<>(n, -n));
S


### `reduce(accumulator)`
The operation `reduce` combines all elements into a single value.
Mathematically, this is similar to operations such as summation or product:

$$
\sum_{x \in A} x
\qquad\text{or}\qquad
\prod_{x \in A} x
$$

More generally, `reduce` repeatedly combines an accumulator with the next element.

In [ ]:
var costs = new RecursiveSet<>(10, 20, 30);

// We start with an initial value of 0, and add each element to the accumulator
var totalSum = costs.reduce(0, (acc, current) -> acc + current);

System.out.println("Set of costs: " + costs);
System.out.println("Total sum of costs: " + totalSum);

## Streams

As `RecursiveSet` implements the interface `Set`, all the methods of the Java collection framework are available.  In particular, we can use *streams*.  The method `RecursiveSet.collector()` collects the elements of a stream into a `RecursiveSet`.

In [ ]:
costs.stream().mapToInt(x -> x).sum()

In [ ]:
vocabulary.stream()
          .map(word -> word.charAt(0))
          .collect(RecursiveSet.collector())

**Summary:**
1. **Uniqueness:** Use `RecursiveSet` when you need a mathematical set without duplicates.
2. **Freeze Contract:** A `RecursiveSet` becomes immutable as soon as it is inserted into another collection.  Use `clone()` to get a mutable copy.
3. **Sequences:** Use records (e.g. the record `Pair`) when you need an ordered, immutable pair or sequence.
4. **Mappings:** Use a `HashMap` when you need to assign a value to a complex key.  Records and instances of `RecursiveSet` can be used as keys.